# Value Function Approximation

_Reinforcement Learning_

**When there are too many states to list one-by-one, fit a function to the value instead of a table.**

A table stores one value per state: $V(s_1), V(s_2), \dots$ — one knob per cell. With a million states you need a
       million knobs, and you must visit each cell many times to learn its knob. For a continuous state (a real number) there
       is no finite list at all.

       Value function approximation throws away the table and fits a parameterized function
       $\hat V(s;w) \approx V^\pi(s)$ instead — a function of the state $s$ controlled by a small weight vector $w$. We tune $w$
       (a few numbers, not a million) by gradient descent so $\hat V$ matches the true value. Because the same $w$ is used for every
       state, learning about one state automatically adjusts the prediction for similar states: the function generalizes.

---

This notebook is a practice scaffold for the **Value Function Approximation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — numpy (runs in Colab; no gym needed)

In [ ]:
import numpy as np

# ---- A small CONTINUOUS environment: a 1-D corridor --------------------
# State is a real number x in [0, 1]. Each step moves right by 0.1.
# Reward is +1 on the step that reaches the goal (x >= 1), else 0.  gamma = 0.9.
# There is NO table: x is real-valued, so we approximate the value instead.
gamma, step = 0.9, 0.1

# ---- Features x(s): 4 radial basis functions (RBFs) + a bias term ------
centers = np.array([0.0, 0.33, 0.66, 1.0]); sigma = 0.25
def x_features(x):
    rbf = np.exp(-((x - centers) ** 2) / (2 * sigma ** 2))
    return np.concatenate([rbf, [1.0]])            # length-5 feature vector

def V_hat(x, w):
    return float(w @ x_features(x))                # linear approximator  w^T x(s)

# ---- Semi-gradient TD(0) -----------------------------------------------
rng = np.random.default_rng(0)
w = np.zeros(5)                                     # the weight vector we learn
alpha = 0.05                                        # step size

for episode in range(600):
    x = rng.uniform(0.0, 0.9)                       # random start state
    done = False
    while not done:
        xp = x + step
        if xp >= 1.0:
            r, v_next, done = 1.0, 0.0, True         # terminal: bootstrap value is 0
        else:
            r, v_next = 0.0, V_hat(xp, w)            # bootstrap off our own estimate
        delta = r + gamma * v_next - V_hat(x, w)     # TD error  delta_t
        # semi-gradient TD update: grad of a LINEAR V_hat is just x_features(x).
        # We do NOT differentiate through the target v_next  ->  "semi"-gradient.
        w = w + alpha * delta * x_features(x)
        x = xp

print("learned weights w:", np.round(w, 3))
for x in [0.0, 0.3, 0.6, 0.9]:
    print(f"  V_hat({x:.1f}) = {V_hat(x, w):.3f}")
# Expected: weights settle and predicted values rise toward the goal, e.g.
#   V_hat(0.0) ~ 0.39, V_hat(0.3) ~ 0.51, V_hat(0.6) ~ 0.69, V_hat(0.9) ~ 0.96

## Visualize the data & results

_As semi-gradient TD(0) trains a LINEAR approximator on a 1-D corridor, does the value error fall — and does the fitted line match the true value?_

In [ ]:
import numpy as np

# 1-D corridor: state x in [0,1]; step +0.1; reward +1 on reaching x>=1; gamma=0.9.
gamma, step = 0.9, 0.1

# True value of a state = gamma^(steps_to_goal - 1) under "always step right".
def true_V(x):
    if x >= 1.0: return 0.0
    steps = int(np.ceil((1.0 - x - 1e-9) / step))
    return gamma ** (steps - 1)

# Features x(s): 4 radial basis functions (RBFs) + bias.
centers = np.array([0.0, 0.33, 0.66, 1.0]); sigma = 0.25
def feats(x):
    return np.concatenate([np.exp(-((x - centers) ** 2) / (2 * sigma ** 2)), [1.0]])

grid = np.linspace(0, 0.95, 20)
Vtrue_grid = np.array([true_V(g) for g in grid])

rng = np.random.default_rng(0)
w = np.zeros(5); alpha = 0.05; errors = []
for ep in range(600):
    x = rng.uniform(0, 0.9); done = False
    while not done:
        xp = x + step
        if xp >= 1.0: r, vnext, done = 1.0, 0.0, True
        else:         r, vnext       = 0.0, w @ feats(xp)
        delta = r + gamma * vnext - w @ feats(x)     # TD error
        w = w + alpha * delta * feats(x)             # semi-gradient TD(0)
        x = xp
    Vhat = np.array([w @ feats(g) for g in grid])
    errors.append(np.sqrt(np.mean((Vhat - Vtrue_grid) ** 2)))   # RMS value error

idx = np.linspace(0, 599, 30).astype(int)            # subsample to 30 points
print("error points:", [[int(i), round(errors[i], 4)] for i in idx])
fx = np.linspace(0, 0.9, 10)
print("learned:", [round(float(w @ feats(x)), 3) for x in fx])
print("true:   ", [round(true_V(x), 3) for x in fx])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A linear approximator has features $x(s) = [0, 1, 0]^\top$ and weights $w = [2, 1, -3]^\top$. The agent gets reward $r = 0$ and lands in $s'$ with $\hat V(s';w) = 4$; use $\gamma = 0.5$ and $\alpha = 0.1$. Compute the prediction, the TD error $\delta_t$, and the updated weights.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Predict $\hat V(s;w) = w^\top x(s) = 2\cdot0 + 1\cdot1 + (-3)\cdot0 = 1$. — _Only the active feature (the middle one, value $1$) contributes; the rest are zeroed out._
- TD target $= r + \gamma\hat V(s';w) = 0 + 0.5\cdot4 = 2$. — _One real reward plus the discounted estimate of the next state._
- TD error $\delta_t = 2 - 1 = 1$. — _We under-estimated by $1$, so the weights should rise where the features are active._
- Update $w \leftarrow w + \alpha\,\delta_t\,x(s) = [2,1,-3] + 0.1\cdot1\cdot[0,1,0] = [2, 1.1, -3]$. — _For a linear approximator $\nabla_w\hat V = x(s)$, so only the active feature's weight changes._

**Answer:** $\hat V(s;w)=1$, $\delta_t = 1$, and $w \leftarrow [2,\,1.1,\,-3]$. Only the second weight moved because only the second feature was active.

</details>

**Problem 2.** Name the three legs of the DEADLY TRIAD and explain why removing any one restores stability.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- List the legs: (1) function approximation, (2) bootstrapping, (3) off-policy training. — _These are the exact three ingredients Sutton & Barto identify; all three together can make the weights diverge._
- Drop function approximation (use a table) → tabular TD converges. — _A table has an independent value per state, so an update to one state cannot blow up another._
- Drop bootstrapping (use Monte-Carlo returns) → the target is a real sampled return, not your own estimate. — _Without a self-referential target there is no feedback loop to amplify error._
- Drop off-policy training (stay on-policy) → on-policy semi-gradient TD with linear features is guaranteed to converge (to the TD fixed point). — _On-policy sampling weights states by how often the policy actually visits them, which keeps the update operator a contraction._

**Answer:** The triad is function approximation + bootstrapping + off-policy training. Any two are safe; all three together can diverge. Removing any single leg (use a table, use Monte-Carlo returns, or stay on-policy) restores a convergence guarantee.

</details>

**Problem 3.** Why is the update called "semi-gradient" rather than "gradient"? What term is dropped?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the bootstrapped objective error: $\delta_t = \big(r + \gamma\hat V(s';w)\big) - \hat V(s;w)$. — _Both the target and the prediction contain the weights $w$._
- A TRUE gradient of the squared TD error would differentiate BOTH $\hat V(s';w)$ and $\hat V(s;w)$ with respect to $w$. — _The chain rule applies to every appearance of $w$._
- The semi-gradient update differentiates ONLY the prediction $\hat V(s;w)$ and treats the target as a fixed label, dropping the $-\gamma\,\nabla_w\hat V(s';w)$ term. — _We want the prediction to chase the target, not the target to chase the prediction; keeping the dropped term often makes learning worse and slower._

**Answer:** It is "semi" because we differentiate only $\hat V(s;w)$ and not the bootstrapped target $r+\gamma\hat V(s';w)$, even though the target also depends on $w$. The dropped term is $-\gamma\,\nabla_w\hat V(s';w)$.

</details>